## 4. Feature Engineering <a id="4"></a>

Raw data rarely has the signal shape models need. We engineer features that encode:
- **Time-of-day risk** — night-time transactions are higher risk
- **Amount-to-balance ratio** — unusual proportion of balance used signals anomaly  
- **Log-transforms** — reduce skew in monetary columns so tree splits are more balanced  
- **Encoded categoricals** — machine-readable labels for all string columns  

>  **Data leakage rule:** all engineering uses only the raw columns.  
> No target statistics (mean-encoding) are applied globally — only on train folds in production.

## Setup & Import

In [17]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder

SEED = 42
np.random.seed(SEED)

## Load prossed data

In [18]:
df = pd.read_csv("data/data_after_eda.csv")
print(f"Loaded data: {df.shape}")

Loaded data: (200000, 24)


##  Datetime Feature Extraction

In [21]:
df["_date"] = pd.to_datetime(df["Transaction_Date"], dayfirst=True, errors="coerce")
df["_time"] = pd.to_datetime(df["Transaction_Time"], format="%H:%M:%S", errors="coerce")

df["Hour"] = df["_time"].dt.hour
df["Minute"] = df["_time"].dt.minute
df["DayOfWeek"] = df["_date"].dt.dayofweek
df["Month"] = df["_date"].dt.month
df["DayOfMonth"] = df["_date"].dt.day

df["IsNight"] = ((df["Hour"] >= 22) | (df["Hour"] <= 5)).astype(int)
df["IsWeekend"] = (df["DayOfWeek"] >= 5).astype(int)
df["IsRushHour"] = (
    ((df["Hour"] >= 7) & (df["Hour"] <= 9)) |
    ((df["Hour"] >= 17) & (df["Hour"] <= 20))
).astype(int)

df.drop(columns=["_date", "_time"], inplace=True)

print("Datetime features created")
print(f"New features: Hour, Minute, DayOfWeek, Month, DayOfMonth, IsNight, IsWeekend, IsRushHour")

Datetime features created
New features: Hour, Minute, DayOfWeek, Month, DayOfMonth, IsNight, IsWeekend, IsRushHour


##  Ratio & Log Transform Features

In [22]:
df["Amt_to_Balance"] = df["Transaction_Amount"] / (df["Account_Balance"] + 1e-6)
df["Log_Amount"] = np.log1p(df["Transaction_Amount"])
df["Log_Balance"] = np.log1p(df["Account_Balance"])
df["Amt_to_Bal_Clipped"] = df["Amt_to_Balance"].clip(upper=5.0)

## Categorical Encoding

In [23]:
ENCODE_COLS = [
    "Gender", "Account_Type", "Transaction_Type",
    "Merchant_Category", "Device_Type", "State"
]

le = LabelEncoder()
for col in ENCODE_COLS:
    df[col + "_enc"] = le.fit_transform(df[col].astype(str))
    print(f"{col:<25} → {col}_enc ({df[col].nunique()} categories)")

print("\n Label encoding complete")

Gender                    → Gender_enc (2 categories)
Account_Type              → Account_Type_enc (3 categories)
Transaction_Type          → Transaction_Type_enc (5 categories)
Merchant_Category         → Merchant_Category_enc (6 categories)
Device_Type               → Device_Type_enc (4 categories)
State                     → State_enc (34 categories)

 Label encoding complete


## Define Final Feature Set

In [24]:
FEATURES = [
    "Transaction_Amount", "Account_Balance", "Age",
    "Amt_to_Balance", "Amt_to_Bal_Clipped",
    "Log_Amount", "Log_Balance",
    "Hour", "Minute", "DayOfWeek", "Month", "DayOfMonth",
    "IsNight", "IsWeekend", "IsRushHour",
    "Gender_enc", "Account_Type_enc", "Transaction_Type_enc",
    "Merchant_Category_enc", "Device_Type_enc", "State_enc",
]
TARGET = "Is_Fraud"

print(f"Total features: {len(FEATURES)}")
print(f"Feature list: {FEATURES}")
print(f"\nTarget column: {TARGET}")

Total features: 21
Feature list: ['Transaction_Amount', 'Account_Balance', 'Age', 'Amt_to_Balance', 'Amt_to_Bal_Clipped', 'Log_Amount', 'Log_Balance', 'Hour', 'Minute', 'DayOfWeek', 'Month', 'DayOfMonth', 'IsNight', 'IsWeekend', 'IsRushHour', 'Gender_enc', 'Account_Type_enc', 'Transaction_Type_enc', 'Merchant_Category_enc', 'Device_Type_enc', 'State_enc']

Target column: Is_Fraud


## Correlation Analysis

In [25]:
corr_cols = FEATURES + [TARGET]
corr = df[corr_cols].corr()

# Correlation with target
target_corr = corr[TARGET].drop(TARGET).sort_values(key=abs, ascending=False)
print("Top 10 features correlated with target:")
print(target_corr.head(10))

Top 10 features correlated with target:
Log_Amount           -0.008023
State_enc             0.005716
IsWeekend            -0.003783
Account_Type_enc     -0.002592
DayOfWeek            -0.002582
IsNight               0.002531
Transaction_Amount   -0.002100
Hour                 -0.001960
Amt_to_Bal_Clipped   -0.001559
Age                  -0.001517
Name: Is_Fraud, dtype: float64


## Save engineered dataset

In [27]:
df.to_csv("data/data_feature_engineered.csv", index=False)
np.save("data/features_list.npy", FEATURES)
print("Feature-engineered data saved")
print(f"Final dataset shape: {df.shape}")

Feature-engineered data saved
Final dataset shape: (200000, 42)
